In [ ]:
""""""""""""""
#The following code is used to automatically take a long highlights cricket video (few minutes) and split it automatically into multiple 1 second clips, each with a single shot
#Refer to the paper description for more detailed understanding of the process

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!pip install ultralytics opencv-python-headless
!pip install opencv-python-headless

import torch
import cv2
import os
from google.colab import files
import tensorflow as tf

from ultralytics import YOLO

In [ ]:
model = YOLO('/content/gdrive/MyDrive/Shot Dataset/Other/Saved Training/Detection/FinalPlayerTypeDetection/weights/best.pt') #Player type detection Model [CHANGE PATH ACCORDINGLY]
ball_model = YOLO('/content/gdrive/MyDrive/ball_detection/best20_30epoch.pt') #Ball detection Model [CHANGE PATH ACCORDINGLY]

In [ ]:
last_frame=[]
last_ball_presence=[]
last_striker_presence=[]
last_bowler_presence=[]
is_batting= [0]
time_frame=[]
shot_type=[]
shot_count=[0]
frame_count=[0]
fps= [0]

In [ ]:
import os
import cv2
import numpy as np


In [ ]:
#This code adds whether bowler, ball and striker is detected in current frame.

def add_to_arrays(frame, result_striker, result_ball):
  last_frame.append(frame)

  striker_detected = False

  for result in result_striker:
        boxes = result.boxes


        for box in boxes:
            cls = int(box.cls[0].cpu().numpy())
            label = model.names[cls]

            if label=='Striker':
              last_striker_presence.append(True)
              striker_detected= True
              break

        if striker_detected == True :
          break

  if striker_detected == False:
    last_striker_presence.append(False)

  bowler_detected = False

  for result in result_striker:
        boxes = result.boxes


        for box in boxes:
            cls = int(box.cls[0].cpu().numpy())
            label = model.names[cls]

            if label=='Bowler':
              last_bowler_presence.append(True)
              bowler_detected= True
              break

        if bowler_detected == True :
          break

  if bowler_detected == False:
    last_bowler_presence.append(False)

  ball_detected = False

  for result in result_ball:
        boxes = result.boxes

        for box in boxes:
            cls = int(box.cls[0].cpu().numpy())
            label = ball_model.names[cls]

            if label=='Ball':
              last_ball_presence.append(True)
              ball_detected= True
              break

        if ball_detected == True :
          break

  if ball_detected == False:
    last_ball_presence.append(False)


In [ ]:
# After first 10 frame of long video, this code does the logic to create a video if necessary conditions of a shot is fullfilled.
# It also removes the last element of the sliding window to keep the sliding window size 10

def add_frame_with_action(output):

  if len(last_frame)==10:
    frame= last_frame[0]

    striker_count = sum(last_striker_presence)
    ball_count = sum(last_ball_presence)
    bowler_detected = last_bowler_presence[:5].count(True) > 1

    print(last_bowler_presence)

    if is_batting[0]==0 and striker_count >=8 and ball_count >=5 and bowler_detected==True:
      is_batting[0]=fps[0]+int(fps[0]/6)

      frame = frame_count[0]

      total_seconds = frame / fps[0]
      minutes = int(total_seconds // 60)
      seconds = int(total_seconds % 60)
      milliseconds = int((total_seconds % 1) * 1000)
      time_frame.append(f"{minutes:02}:{seconds:02}:{milliseconds:03}")

    if is_batting[0]>1 and is_batting[0]<=fps[0]:
      output.write(frame)

    if is_batting[0] >0 :
      is_batting[0] -=1

    last_frame.pop(0)
    last_striker_presence.pop(0)
    last_ball_presence.pop(0)
    last_bowler_presence.pop(0)


In [ ]:
#Main code

import shutil
from google.colab import files

#we require this range because each of our long highlights video were named vid{x} where x is an integer, the code loops for these ranges of highlights video, u can change
range_start = 471
range_end = 480

for vidNum in range(range_start, range_end+1):   #This is automated loop to work for multiple highlights video
  video_type = 'mp4'
  video_name = f'vid{vidNum}'
  video_folder = f'/content/gdrive/MyDrive/Shot Dataset/Videos/Full Length'   #root path location where all highlights video are located

  video_path = f'{video_folder}/{video_name}.{video_type}'

  cap = cv2.VideoCapture(video_path)

  fps=[0]
  fps[0] = int(cap.get(cv2.CAP_PROP_FPS))

  last_frame=[]
  last_ball_presence=[]
  last_striker_presence=[]
  last_bowler_presence=[]
  is_batting= [0]
  time_frame=[]
  shot_type=[]
  shot_count=[0]
  frame_count=[0]

  frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

  output_folder = f'/content/gdrive/MyDrive/Shot Dataset/Videos/Shot Split/{video_name}'  #where videos would be saved, video name is vid{no}_{clip serial for highlight vid_no}
  os.makedirs(output_folder, exist_ok=True)

  output_filetype = 'avi'
  output_filename = os.path.join(output_folder, f'{video_name}_{shot_count[0]}.{output_filetype}')
  out = cv2.VideoWriter(output_filename, cv2.VideoWriter_fourcc('M','J','P','G'), 30, (frame_width, frame_height))

  while cap.isOpened():
      ret, frame = cap.read()

      if not ret:
          break

      results = model(source=frame, max_det=20, conf=0.25, verbose=False)
      results2 = ball_model(source=frame, max_det=1, conf=0.20, verbose=False)

      add_to_arrays(frame= frame, result_striker= results, result_ball= results2)
      add_frame_with_action(output=out)

      if is_batting[0]==1:
        out.release()

        shot_count[0]+=1

        output_filename = os.path.join(output_folder, f'{video_name}_{shot_count[0]}.{output_filetype}')
        out = cv2.VideoWriter(output_filename, cv2.VideoWriter_fourcc('M','J','P','G'), 30, (frame_width, frame_height))

      frame_count[0]+=1

  cap.release()
  out.release()
  os.remove(output_filename)
  cv2.destroyAllWindows()
